<p align="center"><img src="https://raw.githubusercontent.com/reddypranith80-max/ghost-ai/main/public/brand/ghost-ai-logo.svg" width="110" alt="Ghost AI logo"></p>

# Ghost Notebook

A hands-on tour of **Ghost AI** from Python. You will start the app, call every API route, create a project, run the agent pipeline step by step, look at tasks and the git log, export the project to a local folder and draw a progress chart.

Links: [Live demo](https://reddypranith80-max.github.io/ghost-ai/) | [Repository](https://github.com/reddypranith80-max/ghost-ai) | [Docs index](https://github.com/reddypranith80-max/ghost-ai/blob/main/docs/README.md)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reddypranith80-max/ghost-ai/blob/main/notebooks/ghost-notebook.ipynb)

> Colab note: Colab VMs have Node.js, so the setup cells can work there, but the first `npm install` and `npm run build` take a few minutes.

## What is Ghost AI?

You give Ghost AI a goal and a tech stack. It creates a project with a task plan, then walks it through a pipeline of **8 lead agents** (Architect, Backend, Frontend, DevOps, QA, Security, Docs, Reviewer), each with **3 side agents**. Every step moves a task forward and records a commit in a git-style log.

**Honest note:** the agents are *simulated*. No AI model is called; tasks and commits come from deterministic code in the app. This notebook talks to the app's JSON API.

## ELI5: explain it like I am 5

Imagine a toy factory with eight helpers. One draws the plan, one builds the engine, one paints it, one packs the box, one presses every button to test it, one looks for dangers, one writes the instructions, and the last one says "OK, ship it". You tell the factory what toy you want. Then you press a big button, **Run next step**, and each helper takes a turn. In this notebook *you* press the button from Python.

## 1. Setup

You need **Python 3.9+**, **Node.js 20+ (24 recommended)**, **git** and about 2 GB of disk for `node_modules`.

The notebook can do the following for you (next cells), or you can do it by hand in a terminal:

```bash
git clone https://github.com/reddypranith80-max/ghost-ai.git
cd ghost-ai
npm install          # or: npm ci
npm run build
npm start            # serves on http://localhost:3000
```

If the server is already running somewhere, skip all of that and set the environment variable `GHOST_BASE_URL` (for example `http://localhost:3000`) before starting Jupyter. The notebook then only calls the API.

In [ ]:
import json, os, shutil, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

# Where is the Ghost AI server? Override with the env var GHOST_BASE_URL.
BASE_URL = os.environ.get("GHOST_BASE_URL", "http://localhost:3000").rstrip("/")
print("BASE_URL =", BASE_URL)

try:
    import requests
    HAVE_REQUESTS = True
except ImportError:
    HAVE_REQUESTS = False
try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
print("requests:", HAVE_REQUESTS, "| pandas:", HAVE_PANDAS, "(both optional; stdlib fallbacks are used)")


def api(method, path, body=None, raw=False, timeout=30):
    """Call the Ghost AI API. Returns (status_code, parsed_json_or_text). Never raises on HTTP errors."""
    url = BASE_URL + path
    data = None if body is None else json.dumps(body).encode()
    headers = {"Content-Type": "application/json"} if body is not None else {}
    if HAVE_REQUESTS:
        r = requests.request(method, url, data=data, headers=headers, timeout=timeout)
        status, text = r.status_code, r.text
    else:
        req = urllib.request.Request(url, data=data, headers=headers, method=method)
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                status, text = resp.status, resp.read().decode()
        except urllib.error.HTTPError as e:
            status, text = e.code, e.read().decode()
    if raw:
        return status, text
    try:
        return status, json.loads(text)
    except ValueError:
        return status, text


def show(rows, columns=None, title=None):
    """Print a list of dicts as a table (pandas if available, plain text otherwise)."""
    if title:
        print(f"\n== {title} ==")
    if not rows:
        print("(no rows)")
        return
    columns = columns or list(rows[0].keys())
    if HAVE_PANDAS:
        df = pd.DataFrame(rows)[columns]
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.to_string(index=False))
        return
    widths = [max(len(str(c)), *(len(str(r.get(c, ""))) for r in rows)) for c in columns]
    line = lambda vals: "  ".join(str(v).ljust(w) for v, w in zip(vals, widths))
    print(line(columns))
    print(line(["-" * w for w in widths]))
    for r in rows:
        print(line([r.get(c, "") for c in columns]))


def healthy():
    try:
        status, body = api("GET", "/api/health", timeout=3)
        return status == 200 and isinstance(body, dict) and body.get("ok") is True
    except Exception:
        return False

### Classic shell version (optional)

If you prefer plain shell cells, uncomment these lines. `!` runs a command from Jupyter. They are commented out so that running the whole notebook never re-clones or reinstalls by accident.

In [ ]:
# !git clone https://github.com/reddypranith80-max/ghost-ai.git
# %cd ghost-ai
# !npm install
# !npm run build
# Start the server in a separate terminal with:  npm start     (blocks, so do not run it in a notebook cell)
print("Shell cells are commented out. Use the helper cells below, or run the commands in a terminal.")

### Find (or clone) the repo, then start the server

`ensure_server()` does the following, in order, and never hangs:

1. If `BASE_URL` already answers `/api/health`, it does nothing.
2. Otherwise it finds the repo (this notebook's parent folder, `./ghost-ai`, or clones it), runs `npm install` and `npm run build` when needed, then starts `next start` with `subprocess.Popen` and **waits up to 120 s** for the health check.

It only starts a server for a local `BASE_URL` (localhost); for a remote URL it just waits briefly and reports.

In [ ]:
from urllib.parse import urlparse

SERVER = None  # subprocess handle if this notebook started the server
NPM = "npm.cmd" if os.name == "nt" else "npm"
NPX = "npx.cmd" if os.name == "nt" else "npx"


def find_repo():
    here = Path.cwd().resolve()
    for cand in (here, here.parent, here / "ghost-ai"):
        if (cand / "package.json").exists() and (cand / "src" / "app").exists():
            return cand
    return None


def wait_healthy(seconds=120):
    deadline = time.time() + seconds
    while time.time() < deadline:
        if healthy():
            return True
        if SERVER is not None and SERVER.poll() is not None:
            print("Server process exited early with code", SERVER.returncode)
            return False
        time.sleep(1.5)
    return False


def ensure_server():
    global SERVER
    if healthy():
        print("Server already running at", BASE_URL)
        return True
    u = urlparse(BASE_URL)
    if u.hostname not in ("localhost", "127.0.0.1"):
        print("BASE_URL is remote and not answering; not starting anything.")
        return False
    repo = find_repo()
    if repo is None:
        print("Cloning the repo ...")
        subprocess.run(["git", "clone", "https://github.com/reddypranith80-max/ghost-ai.git"], check=True)
        repo = Path("ghost-ai").resolve()
    print("Repo:", repo)
    if not (repo / "node_modules").exists():
        print("npm install (a few minutes the first time) ...")
        subprocess.run([NPM, "install"], cwd=repo, check=True)
    if not (repo / ".next").exists():
        print("npm run build ...")
        subprocess.run([NPM, "run", "build"], cwd=repo, check=True)
    port = str(u.port or 3000)
    SERVER = subprocess.Popen([NPX, "next", "start", "-p", port], cwd=repo, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    ok = wait_healthy(120)
    print("Server is up." if ok else "Server did not become healthy in time (check the terminal / port).")
    return ok


SERVER_OK = ensure_server()

## 2. Health check

`GET /api/health` reports liveness and whether the database is reachable.

In [ ]:
status, body = api("GET", "/api/health")
print("HTTP", status)
show([body], ["ok", "db", "version", "uptimeSeconds", "authEnabled", "time"], "health")
assert status == 200 and body["ok"], "Server is not healthy"

## 3. The crew: 8 agents x 3 sub-agents

`GET /api/agents` is the single source of truth for the crew.

In [ ]:
status, agents = api("GET", "/api/agents")
show(
    [{"agent": a["name"], "codename": a["codename"], "tools": ", ".join(a["tools"]), "produces": a["produces"]} for a in agents],
    title="8 lead agents",
)
show(
    [{"agent": a["name"], "sub_agent": s["name"], "role": s["role"]} for a in agents for s in a["subAgents"]],
    title="24 sub-agents",
)
print(len(agents), "agents,", sum(len(a["subAgents"]) for a in agents), "sub-agents")

## 4. Create a project

`POST /api/projects` with `name` (2-60 chars), `brief` (10-2000 chars) and `stack` (a list). The response contains the new project with its generated task plan. We remember the id so the cleanup cell can delete it.

In [ ]:
CREATED = []  # project ids created by this notebook

status, stack_catalog = api("GET", "/api/stack")
print("Stack categories:", ", ".join(f"{k} ({len(v)})" for k, v in stack_catalog.items()))

status, project = api("POST", "/api/projects", {
    "name": "Notebook Todo App",
    "brief": "A small todo app with login, created from the Ghost Notebook.",
    "stack": ["Next.js", "Drizzle"],
})
print("HTTP", status)
assert status == 201, project
PROJECT_ID = project["id"]
CREATED.append(PROJECT_ID)
print("Created project", PROJECT_ID, "| status:", project["status"], "| tasks:", len(project["tasks"]))
show(project["tasks"][:8], ["title", "module", "agent", "status", "progress"], "first tasks of the plan")

## 5. Run the pipeline step by step

`POST /api/projects/:id/run` advances one step: a task in `review` becomes `done`, a `running` task moves to `review`, and the next `todo` task starts `running`. Each step may add commits. When everything is done the API answers `409 pipeline_complete`.

In [ ]:
from collections import Counter

HISTORY = []  # (step, counts) for the chart
def counts(tasks):
    c = Counter(t["status"] for t in tasks)
    return {k: c.get(k, 0) for k in ("todo", "running", "review", "done", "failed")}

HISTORY.append((0, counts(project["tasks"])))
STEPS = 6
for step in range(1, STEPS + 1):
    status, detail = api("POST", f"/api/projects/{PROJECT_ID}/run")
    if status == 409:
        print(f"step {step}: pipeline already complete")
        break
    assert status == 200, detail
    c = counts(detail["tasks"])
    HISTORY.append((step, c))
    print(f"step {step}: {c} | project status: {detail['status']} | commits: {len(detail['gitlog'])}")

## 6. Tasks by status

`GET /api/projects/:id/tasks?status=...` filters server-side (also `module` and `agent`).

In [ ]:
for st in ("todo", "running", "review", "done"):
    status, tasks = api("GET", f"/api/projects/{PROJECT_ID}/tasks?status={st}")
    show(tasks, ["title", "module", "agent", "progress"], f"{st} ({len(tasks)})")

status, stats = api("GET", "/api/stats")
print("\nGlobal stats:", stats)

## 7. Git log

`GET /api/projects/:id/gitlog?limit=N` returns the commits the agents recorded.

In [ ]:
status, log = api("GET", f"/api/projects/{PROJECT_ID}/gitlog?limit=10")
show(log, ["hash", "agent", "message", "files", "at"], "latest commits")

## 8. Export to a local folder

`GET /api/projects/:id/export?format=md` returns a Markdown report; `format=json` returns the full project. Below we save both into a local folder, the same idea as the dashboard's "save to folder" feature.

In [ ]:
import re
status, report = api("GET", f"/api/projects/{PROJECT_ID}/export?format=md", raw=True)
assert status == 200
status, as_json = api("GET", f"/api/projects/{PROJECT_ID}/export?format=json", raw=True)
assert status == 200

slug = re.sub(r"[^a-z0-9]+", "-", project["name"].lower()).strip("-") or "project"
EXPORT_DIR = Path("ghost-export") / slug
os.makedirs(EXPORT_DIR, exist_ok=True)
(EXPORT_DIR / f"{slug}.md").write_text(report, encoding="utf-8")
(EXPORT_DIR / f"{slug}.json").write_text(as_json, encoding="utf-8")
(EXPORT_DIR / "gitlog.txt").write_text("\n".join(f"{c['hash']} [{c['agent']}] {c['message']}" for c in log), encoding="utf-8")

for p in sorted(EXPORT_DIR.iterdir()):
    print(f"{p.name:30} {p.stat().st_size:>8} bytes")
print("Saved to", EXPORT_DIR.resolve())
print("\n--- first lines of the Markdown report ---")
print("\n".join(report.splitlines()[:12]))

## 9. Progress chart

A matplotlib stacked bar per step if matplotlib is installed, otherwise a text chart.

In [ ]:
SYMBOL = {"todo": ".", "running": ">", "review": "?", "done": "#", "failed": "x"}
try:
    import matplotlib
    import matplotlib.pyplot as plt
    steps = [s for s, _ in HISTORY]
    bottom = [0] * len(steps)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for key, color in (("done", "#10b981"), ("review", "#8b5cf6"), ("running", "#06b6d4"), ("todo", "#94a3b8")):
        vals = [c[key] for _, c in HISTORY]
        ax.bar(steps, vals, bottom=bottom, label=key, color=color)
        bottom = [b + v for b, v in zip(bottom, vals)]
    ax.set_xlabel("pipeline step"); ax.set_ylabel("tasks"); ax.set_title("Tasks by status after each step"); ax.legend(ncol=4, fontsize=8)
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; text chart (. todo  > running  ? review  # done)\n")
    for s, c in HISTORY:
        print(f"step {s}: " + "".join(SYMBOL[k] * c[k] for k in ("done", "review", "running", "todo", "failed")))

## 10. Try your own brief

Change the values below and run the cell. It creates a project, runs the whole pipeline to the end (with a safety cap) and prints a summary. It is deleted in the cleanup cell.

In [ ]:
MY_NAME = "My Own Idea"
MY_BRIEF = "A recipe sharing site where people can save and rate recipes."
MY_STACK = ["Next.js", "Drizzle", "Clerk"]
MAX_STEPS = 60

status, mine = api("POST", "/api/projects", {"name": MY_NAME, "brief": MY_BRIEF, "stack": MY_STACK})
if status != 201:
    print("Could not create the project:", mine)  # validation errors are explained in the response
else:
    CREATED.append(mine["id"])
    steps = 0
    while steps < MAX_STEPS:
        status, mine = api("POST", f"/api/projects/{mine['id']}/run")
        if status == 409:
            break
        steps += 1
    status, mine = api("GET", f"/api/projects/{CREATED[-1]}")
    print(f"{MY_NAME}: {steps} steps, status={mine['status']}, tasks={counts(mine['tasks'])}, commits={len(mine['gitlog'])}")
    show(mine["gitlog"][:5], ["hash", "agent", "message"], "latest commits")

## Learn more

- Live demo: <https://reddypranith80-max.github.io/ghost-ai/>
- Repository and README: <https://github.com/reddypranith80-max/ghost-ai>
- Docs index: <https://github.com/reddypranith80-max/ghost-ai/blob/main/docs/README.md>
- API reference (OpenAPI JSON from your own server): `GET {BASE_URL}/api/openapi`
- Fork it and make it yours: <https://github.com/reddypranith80-max/ghost-ai/blob/main/docs/FORK-AND-CLONE.md>

## 11. Cleanup

Deletes the projects this notebook created (`DELETE /api/projects/:id`), stops the server **only if this notebook started it**, and optionally removes the exported folder.

In [ ]:
for pid in list(CREATED):
    status, body = api("DELETE", f"/api/projects/{pid}")
    print("delete", pid, "->", status)
    if status in (200, 404):
        CREATED.remove(pid)

if SERVER is not None and SERVER.poll() is None:
    if os.name == "nt":
        subprocess.run(["taskkill", "/PID", str(SERVER.pid), "/T", "/F"], capture_output=True)
    else:
        SERVER.terminate()
    print("Stopped the server started by this notebook.")
else:
    print("No server started by this notebook; nothing to stop.")

REMOVE_EXPORT = False  # set True to delete ./ghost-export
if REMOVE_EXPORT:
    shutil.rmtree("ghost-export", ignore_errors=True)
    print("Removed ghost-export/")